# 🚗 Car Price Prediction — Final Clean Notebook

**Goal:** Predict used-car prices with Linear Regression, Ridge Regression and Lasso Regression.

### Important fixes in this version
- `Price` and `Log_price` are both excluded from the feature matrix to prevent **target leakage**.
- A separate `LabelEncoder` is fitted and saved for every categorical column.
- Train/test split is performed only after the final feature matrix is created.
- RMSE and R² are compared in one table.
- The same preprocessing artifacts used by the Streamlit app are saved at the end.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

BASE_DIR = Path.cwd()


## 2. Load the Dataset

The notebook downloads the same dataset used by the training script. If you already have the CSV locally, you can replace this cell with `pd.read_csv(...)`.

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "1.04. Real-life example.csv"
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "smritisingh1997/car-salescsv",
    file_path,
)

print("Shape:", df.shape)
display(df.head())


## 3. Basic Data Inspection

In [ ]:
display(df.info())
display(df.describe(include="all").T)
display(df.isnull().sum().sort_values(ascending=False).head(20))


## 4. Clean the Data

Rows without `Price` or `EngineV` are removed. Engine volumes above 10 L are treated as unrealistic outliers for this dataset.

In [ ]:
df = df.dropna(subset=["Price", "EngineV"]).copy()
df = df[df["EngineV"] <= 10].copy()

print("Cleaned shape:", df.shape)


## 5. Transform the Target

Car prices are right-skewed, so we model `log(Price)` and convert predictions back with `exp()`.

In [ ]:
df["Log_price"] = np.log(df["Price"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["Price"], bins=50)
axes[0].set_title("Original Price")
axes[1].hist(df["Log_price"], bins=50)
axes[1].set_title("Log-transformed Price")
plt.tight_layout()
plt.show()


## 6. Encode Categorical Features

Each categorical column gets its **own** `LabelEncoder`. This is important because the Streamlit app needs to reproduce the exact mapping used during training.

In [ ]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
encoders = {}

for col in cat_cols:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col])
    encoders[col] = encoder

print("Categorical columns:", cat_cols)


## 7. Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()


## 8. Build Features and Target — Leakage Check

**Critical:** `Price` is the original target and `Log_price` is the transformed target. Neither can be used as an input feature. Including `Price` would leak the answer into the model.

In [ ]:
X = df.drop(columns=["Price", "Log_price"])
y = df["Log_price"]

feature_columns = X.columns.tolist()
print("Features:", feature_columns)
print("Target:", y.name)
assert "Price" not in X.columns
assert "Log_price" not in X.columns


## 9. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


## 10. Train Linear, Ridge and Lasso

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0, random_state=42),
}

fitted = {}
results = []

for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    preds = estimator.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    fitted[name] = estimator
    results.append({"Model": name, "RMSE": rmse, "R2": r2})

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df)


## 11. Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, estimator) in zip(axes, fitted.items()):
    preds = estimator.predict(X_test)
    ax.scatter(y_test, preds, alpha=0.55, edgecolor="k", linewidth=0.2)
    lo = min(y_test.min(), preds.min())
    hi = max(y_test.max(), preds.max())
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=2)
    ax.set_xlabel("Actual log price")
    ax.set_ylabel("Predicted log price")
    ax.set_title(name)

plt.tight_layout()
plt.show()


## 12. Select the Lowest-RMSE Model

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted[best_name]
print("Selected model:", best_name)


## 13. Save Deployment Artifacts

These files are consumed by `app.py`. The saved model and encoders are kept consistent with the deployment environment.

In [ ]:
# If this notebook is inside the project directory, artifacts are saved there.
joblib.dump(best_model, BASE_DIR / "car_price_model.pkl")
joblib.dump(encoders, BASE_DIR / "encoders.pkl")
joblib.dump(cat_cols, BASE_DIR / "cat_cols.pkl")
joblib.dump(feature_columns, BASE_DIR / "feature_columns.pkl")

numeric_cols = [c for c in feature_columns if c not in cat_cols]
numeric_ranges = {
    c: (float(df[c].min()), float(df[c].max()))
    for c in numeric_cols
}
joblib.dump(numeric_ranges, BASE_DIR / "numeric_ranges.pkl")
results_df.to_csv(BASE_DIR / "model_results.csv", index=False)

print("Artifacts saved successfully.")


## 14. Final Checklist

- ✅ No `Price` leakage into `X`
- ✅ No `Log_price` leakage into `X`
- ✅ Separate encoder for every categorical feature
- ✅ Same feature order saved for the web app
- ✅ Numeric ranges saved for safe input controls
- ✅ Model comparison includes RMSE and R²
- ✅ Best model saved for Streamlit deployment